# MiniKernel on a GPU (Google Colab)

1. Runtime → Change runtime type → T4 GPU.
2. Clone this repo, then install the package.
3. Compile a kernel to PTX and launch it with CuPy.

In [ ]:
!nvidia-smi --query-gpu=name,compute_cap --format=csv

In [ ]:
from pathlib import Path
import numpy as np

from minikernel.frontend import compile_file
from minikernel.codegen_ptx import compile_to_ptx
from minikernel.interpreter import interpret

kernel = compile_file("examples/add_one.mk")
ptx = compile_to_ptx(kernel)
Path("add_one.ptx").write_text(ptx)
print(ptx[:400])

In [ ]:
import cupy as cp

n = 8
host_in = np.arange(n, dtype=np.int32) * 10
ref = interpret(kernel, {"a": host_in.tolist()}, n)

module = cp.RawModule(path="add_one.ptx")
fn = module.get_function("add_one")
data = cp.asarray(host_in)
out = cp.full(n, 0x7EADBEEF, dtype=cp.int32)
fn((1,), (32,), (data, out, np.uint32(n)))
cp.cuda.Device().synchronize()
print("gpu", out.get())
print("ref", ref["c"])